# Lecture 3 (optional): Which atoms are crystalline? Descriptors and clustering in a Lennard-Jones system

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Edinburgh-Chemistry-Teaching/AdvancedPhysicalChemistry/blob/main/L3_representations/L3_descriptors_clustering_LJ.ipynb)

<a rel="license" href="https://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons Licence" style="border-width:0" src="https://licensebuttons.net/l/by-sa/4.0/88x31.png" title='This work is licensed under a Creative Commons Attribution-ShareAlike 4.0 International License.' align="right"/></a>

Author: Dr Antonia Mey
Email: antonia.mey@ed.ac.uk

**Time needed:** about 40 minutes. This notebook is **optional** practice for Lecture 3 (representing chemistry, clustering).

Solutions are in the green **Click here to see solution** drop-downs below each task. Try the task yourself first, then open the drop-down to check.

### Learning objectives
By the end of this notebook, you should be able to
* Turn atomic positions into descriptors that do not depend on how the system is moved, rotated or numbered: the radial distribution function $g(r)$, coordination numbers, per-atom energies and the hexatic order $|\psi_6|$.
* Explain why the choice of cutoff matters, and why a hard cutoff gives a descriptor that jumps.
* Cluster atoms with k-means, explain why features are standardised first, and use an elbow plot and the silhouette score to choose the number of clusters.
* Run DBSCAN and explain when it finds separate groups and when it does not.

### Table of Contents

0. [The data: one snapshot of 855 atoms](#0-data)
1. [Distances and the radial distribution function](#1-gr)
2. [Coordination number](#2-cn)
3. [More descriptors: energy and hexatic order](#3-descriptors)
4. [k-means: finding solid and liquid](#4-kmeans)
5. [How many clusters?](#5-k)
6. [DBSCAN](#6-dbscan)
7. [Key points](#7-key-points)

### The data

This is the system from the lecture: 855 atoms in two dimensions that interact through the Lennard-Jones potential
$$U(r) = 4\varepsilon\left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^{6}\right].$$
A crystal was placed in the middle of the box and the outside was melted, then everything was run with molecular dynamics. The file contains the positions of all atoms in one snapshot, in **reduced units** (lengths in units of $\sigma$, energies in units of $\varepsilon$). The box is **periodic** which means an atom leaving on the right comes back on the left.

**The idea of this notebook:** the file contains *only* positions. Nobody has told us which atoms are crystalline. We will build descriptors for each atom and let clustering find the groups.

### How to open this notebook

* **Noteable (recommended):** open it from the Lecture 3 page on Learn. Everything you need is already installed.
* **Google Colab (backup):** click the *Open in Colab* button at the top. You need a Google account. Save a copy to your Drive (*File → Save a copy in Drive*) or your answers will be lost.
* **Your own computer:** you need `numpy`, `pandas`, `matplotlib` and `scikit-learn` (see `requirements.txt` in the repository).

**<span style="color:black">Jupyter Cheat Sheet</span>**
- To run the currently highlighted cell and move focus to the next cell, hold <kbd>&#x21E7; Shift</kbd> and press <kbd>&#x23ce; Enter</kbd>;
- To get help for a specific function, place the cursor within the function's brackets, hold <kbd>&#x21E7; Shift</kbd>, and press <kbd>&#x21E5; Tab</kbd>;
- Cells marked `# FIXME` are for you to complete.

# Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import EllipseCollection

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#b0b0b0"

# The data file lives in the data/ folder of the course repository.
# If it is not found locally (e.g. on Colab), it is downloaded from GitHub.
DATA_URL = "https://raw.githubusercontent.com/Edinburgh-Chemistry-Teaching/AdvancedPhysicalChemistry/main/data/"


def load(name):
    try:
        return pd.read_csv("../data/" + name)
    except FileNotFoundError:
        return pd.read_csv(DATA_URL + name)


def plot_atoms(ax, colours, title="", cmap=None, vmin=None, vmax=None):
    """Draw every atom as a disc of diameter 1 sigma. `colours` is one colour per atom,
    or one number per atom together with a colour map `cmap`."""
    n = len(pos)
    discs = EllipseCollection(np.ones(n), np.ones(n), np.zeros(n), units="xy", offsets=pos,
                              offset_transform=ax.transData, edgecolors="white", linewidths=0.3)
    if cmap is None:
        discs.set_facecolor(colours)
    else:
        discs.set_array(np.asarray(colours)); discs.set_cmap(cmap); discs.set_clim(vmin, vmax)
    ax.add_collection(discs)
    ax.set_xlim(0, box[0]); ax.set_ylim(0, box[1]); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title)
    return discs

# 0. The data: one snapshot of 855 atoms <a id="0-data"></a>

In [ ]:
df = load("lj_2d_snapshot.csv")
pos = df[["x", "y"]].values             # positions / sigma, shape (N, 2)
box = np.array([45.3078, 23.5426])      # box lengths / sigma (periodic in x and y)
N = len(pos)
print(N, "atoms; number density rho =", round(N / box.prod(), 3), "per sigma^2")

fig, ax = plt.subplots(figsize=(12, 6))
plot_atoms(ax, GREY, "855 Lennard-Jones atoms: no labels")
plt.show()

By eye you can probably see an ordered region in the middle and disordered regions at the sides. The raw representation of this snapshot is $2 \times 855 = 1710$ numbers, and they would all change if we shifted the box or renumbered the atoms. Instead we describe each atom by its **surroundings**.

# 1. Distances and the radial distribution function <a id="1-gr"></a>

Distances between atoms do not change when the system is moved or rotated. Because the box is periodic, the distance between two atoms is measured to the *nearest* copy of the other atom (the **minimum image**). The function below returns, for every pair of atoms $i$, $j$, the vector from $i$ to $j$ and its length.

In [ ]:
def pair_vectors(pos, box):
    """Vectors d[i, j] from atom i to atom j (minimum image) and distances r[i, j].
    The distance of an atom to itself is set to infinity so it never counts as a neighbour."""
    d = pos[None, :, :] - pos[:, None, :]
    d -= box * np.round(d / box)         # nearest periodic copy
    r = np.hypot(d[..., 0], d[..., 1])
    np.fill_diagonal(r, np.inf)
    return d, r


d, r = pair_vectors(pos, box)
print("r has shape", r.shape, "; smallest distance:", r.min().round(3), "sigma")

The **radial distribution function** $g(r)$ tells us how likely it is to find another atom at distance $r$ from an atom, compared with a completely random (ideal gas) arrangement at the same density. In 2D, for bins of width $\Delta r$:
$$g(r) = \frac{\text{number of pairs } (i, j) \text{ with distance in } [r, r + \Delta r)}{N \, \rho \, \pi \left[(r + \Delta r)^2 - r^2\right]},$$
where every pair is counted twice ($i \to j$ and $j \to i$), $\rho = N / (L_x L_y)$ is the number density and the denominator is the number of atoms an ideal gas would put in each ring.

## Tasks 1 <a id="tasks-1"></a>
1. Compute $g(r)$ from 0 to $4\sigma$ in bins of $0.05\sigma$ (use `np.histogram` on the finite entries of `r`, i.e. `r[r < 4]`) and plot it.
2. Where is the first peak? Between which distances is $g(r)$ low after the first peak (the first minimum)? Mark $r = 1.5\sigma$ with `ax.axvline`.

In [ ]:
# 1. and 2. Radial distribution function
# FIXME

<details><summary {style="color:green;font-weight:bold"}> Click here to see solution to Task 1.1 </summary>

```python
# 1. and 2. Radial distribution function
dr = 0.05
edges = np.arange(0, 4 + dr, dr)
counts, _ = np.histogram(r[r < 4], bins=edges)
rho = N / box.prod()
g = counts / (N * rho * np.pi * (edges[1:] ** 2 - edges[:-1] ** 2))
r_mid = 0.5 * (edges[1:] + edges[:-1])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(r_mid, g, color=BLUE)
ax.axhline(1, color="grey", lw=0.5)
ax.axvline(1.5, color=ORANGE, ls="--", label="r = 1.5 sigma")
ax.set_xlabel("r / sigma"); ax.set_ylabel("g(r)"); ax.legend()
plt.show()
print("first peak at r =", r_mid[np.argmax(g)].round(2), "sigma")
```

</details>

*Your answers to question 2 here.*

<details><summary {style="color:green;font-weight:bold"}> Click here to see answers to question 2 (Tasks 1) </summary>

The first peak is at about $1.1\sigma$, the distance of nearest neighbours (close to the minimum of the LJ potential at $2^{1/6}\sigma = 1.12\sigma$). After it $g(r)$ drops and stays low between about $1.45\sigma$ and $1.7\sigma$ before the second shell starts. A cutoff anywhere in this first minimum, such as $1.5\sigma$, separates "nearest neighbours" from "the rest". $g(r)$ goes back to 1 at large $r$: far away, the positions of other atoms are uncorrelated.

</details>

# 2. Coordination number <a id="2-cn"></a>

The **coordination number** of atom $i$ is the number of atoms closer than a cutoff $r_c$:
$$\mathrm{CN}_i = \text{number of atoms } j \neq i \text{ with } r_{ij} < r_c .$$
It only uses distances, so it does not change when the system is moved, rotated or renumbered. From the lecture: in a perfect 2D crystal every atom has 6 neighbours.

## Tasks 2 <a id="tasks-2"></a>
1. Compute `CN` for every atom with $r_c = 1.5\sigma$ (hint: `(r < 1.5)` is a boolean matrix; sum it along one axis). Print how many atoms have each value (`np.unique(CN, return_counts=True)`) and the mean.
2. Draw the snapshot coloured by CN (`plot_atoms(ax, CN, cmap="Blues", vmin=2, vmax=6)`, then `plt.colorbar(...)` on the object it returns). Where are the atoms with fewer than 6 neighbours?
3. Compute the mean CN for cutoffs of 1.3, 1.4, 1.5, 1.6 and 1.7$\sigma$. How sensitive is CN to this choice?

In [ ]:
# 1. Coordination numbers
# FIXME

<details><summary {style="color:green;font-weight:bold"}> Click here to see solution to Task 2.1 </summary>

```python
# 1. Coordination numbers
CN = (r < 1.5).sum(axis=1)
print(np.unique(CN, return_counts=True))
print("mean CN:", CN.mean().round(2))
```

</details>

In [ ]:
# 2. Snapshot coloured by CN
# FIXME

<details><summary {style="color:green;font-weight:bold"}> Click here to see solution to Task 2.2 </summary>

```python
# 2. Snapshot coloured by CN
fig, ax = plt.subplots(figsize=(12, 6))
im = plot_atoms(ax, CN, "coordination number", cmap="Blues", vmin=2, vmax=6)
plt.colorbar(im, ax=ax, shrink=0.6)
plt.show()
```

</details>

In [ ]:
# 3. Mean CN against the cutoff
# FIXME

<details><summary {style="color:green;font-weight:bold"}> Click here to see solution to Task 2.3 </summary>

```python
# 3. Mean CN against the cutoff
for rc in [1.3, 1.4, 1.5, 1.6, 1.7]:
    print(f"r_c = {rc} sigma: mean CN = {(r < rc).sum(axis=1).mean():.2f}")
```

</details>

*Your answers to questions 2 and 3 here.*

<details><summary {style="color:green;font-weight:bold"}> Click here to see answers to questions 2 and 3 (Tasks 2) </summary>

Most atoms have CN = 6 (539 of 855) or 5 (250); a few have 4 or 3; the mean is 5.54. Atoms with 6 neighbours fill the crystal in the middle, but they are also common in the liquid: this is a dense liquid, so counting neighbours alone does not cleanly separate the two. The mean CN goes from 4.8 at $1.3\sigma$ to 5.9 at $1.7\sigma$: the cutoff is a choice we make, and it changes the answer. Because CN is a whole number, an atom whose neighbour sits right at the cutoff flips between two values as it vibrates (the "smooth" point from the lecture).

</details>

# 3. More descriptors: energy and hexatic order <a id="3-descriptors"></a>

Two more numbers per atom:

* **Potential energy** of atom $i$: half of its interaction with all other atoms (half, so that each pair is shared between its two atoms),
$$E_i = \tfrac12 \sum_{j \neq i} U(r_{ij}), \qquad \text{using only pairs with } r_{ij} < 2.5\sigma .$$
* **Hexatic order** $|\psi_6|$, which looks at the *angles* to the neighbours rather than their number:
$$|\psi_6(i)| = \Big|\frac{1}{\mathrm{CN}_i}\sum_{j} e^{\,6 \mathrm{i} \theta_{ij}}\Big| ,$$
where $\theta_{ij}$ is the angle of the vector from $i$ to neighbour $j$. It is 1 for a perfect hexagon of neighbours and small for a disordered one. The function is given below.

In [ ]:
def hexatic_order(d, r, rc=1.5):
    """|psi_6| of every atom, from the pair vectors d and distances r (neighbours: r < rc)."""
    neighbour = r < rc
    theta = np.arctan2(d[..., 1], d[..., 0])
    z = (np.exp(6j * theta) * neighbour).sum(axis=1)
    return np.abs(z) / np.maximum(neighbour.sum(axis=1), 1)


psi6 = hexatic_order(d, r)

## Tasks 3 <a id="tasks-3"></a>
1. Compute the potential energy `E` of every atom in units of $\varepsilon$ (hint: build a matrix `U` with `np.where(r < 2.5, 4 * (r**-12 - r**-6), 0)`, then sum along one axis).
2. Make two scatter plots side by side: CN against `E`, and CN against `psi6`. Compute the correlation coefficient of each pair with `np.corrcoef`.
3. Draw the snapshot coloured by `psi6` (`cmap="Blues", vmin=0.3, vmax=1`). Which descriptor shows the crystal most clearly, and do CN and energy carry the same information?

In [ ]:
# 1. Potential energy per atom
# FIXME

<details><summary {style="color:green;font-weight:bold"}> Click here to see solution to Task 3.1 </summary>

```python
# 1. Potential energy per atom
U = np.where(r < 2.5, 4 * (r**-12 - r**-6), 0)
E = 0.5 * U.sum(axis=1)
print("mean energy per atom:", E.mean().round(2), "epsilon")
```

</details>

In [ ]:
# 2. Scatter plots and correlations
# FIXME

<details><summary {style="color:green;font-weight:bold"}> Click here to see solution to Task 3.2 </summary>

```python
# 2. Scatter plots and correlations
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(CN, E, s=10, alpha=0.4, color=BLUE)
axes[0].set_ylabel("potential energy / epsilon")
axes[1].scatter(CN, psi6, s=10, alpha=0.4, color=BLUE)
axes[1].set_ylabel("|psi_6|")
for ax in axes:
    ax.set_xlabel("coordination number")
plt.show()
print("r(CN, E)    =", np.corrcoef(CN, E)[0, 1].round(2))
print("r(CN, psi6) =", np.corrcoef(CN, psi6)[0, 1].round(2))
```

</details>

In [ ]:
# 3. Snapshot coloured by |psi_6|
# FIXME

<details><summary {style="color:green;font-weight:bold"}> Click here to see solution to Task 3.3 </summary>

```python
# 3. Snapshot coloured by |psi_6|
fig, ax = plt.subplots(figsize=(12, 6))
im = plot_atoms(ax, psi6, "hexatic order |psi_6|", cmap="Blues", vmin=0.3, vmax=1)
plt.colorbar(im, ax=ax, shrink=0.6)
plt.show()
```

</details>

*Your answers to question 3 here.*

<details><summary {style="color:green;font-weight:bold"}> Click here to see answers to question 3 (Tasks 3) </summary>

CN and energy are strongly correlated ($r \approx -0.69$): more neighbours means more attractive pairs and a lower energy, so the two carry much of the same information. CN and $|\psi_6|$ are only moderately correlated ($r \approx 0.42$): $|\psi_6|$ measures the arrangement of the neighbours, not how many there are. Atoms with 6 neighbours can have anything from a perfect hexagon to a squashed, irregular ring. The $|\psi_6|$ snapshot shows the crystal most clearly. (Your values differ a little from the lecture slides, which averaged each descriptor over a short stretch of the simulation; here we use one snapshot.)

</details>

# 4. k-means: finding solid and liquid <a id="4-kmeans"></a>

Each atom is now a point in a **feature space**. We start with the two features used in the lecture, CN and energy, and ask k-means for $K = 2$ clusters. k-means uses distances, so features with large numbers would dominate; `StandardScaler` rescales each feature to mean 0 and standard deviation 1.

To check the result we use a reference that k-means never sees: we call an atom *ordered* if $|\psi_6| > 0.6$.

In [ ]:
ordered = psi6 > 0.6
print(f"{ordered.mean():.0%} of atoms have |psi_6| > 0.6")


def solid_like(labels):
    """k-means numbers its clusters arbitrarily: call the cluster with the lower mean energy 'solid-like'."""
    solid = 0 if E[labels == 0].mean() < E[labels == 1].mean() else 1
    return labels == solid

## Tasks 4 <a id="tasks-4"></a>
1. Standardise the features `np.c_[CN, E]`, run `KMeans(n_clusters=2, n_init=10, random_state=0)`, and use `solid_like` to get a boolean array. Draw the snapshot with solid-like atoms in orange and liquid-like atoms in blue (`np.where(solid, ORANGE, BLUE)`). What fraction of atoms is solid-like, and for what fraction does it agree with `ordered`?
2. For argon, $\varepsilon / k_\mathrm{B} = 120$ K. Convert the energy to kelvin (`E_K = 120 * E`) and repeat the clustering on `np.c_[CN, E_K]` **without** standardising. How does the agreement change, and why? Then standardise and check again.

In [ ]:
# 1. k-means on standardised (CN, E)
# FIXME

<details><summary {style="color:green;font-weight:bold"}> Click here to see solution to Task 4.1 </summary>

```python
# 1. k-means on standardised (CN, E)
F = StandardScaler().fit_transform(np.c_[CN, E])
labels = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(F)
solid = solid_like(labels)

fig, ax = plt.subplots(figsize=(12, 6))
plot_atoms(ax, np.where(solid, ORANGE, BLUE), "k-means on (CN, E): solid-like orange, liquid-like blue")
plt.show()
print(f"solid-like: {solid.mean():.2f}; agreement with |psi_6| > 0.6: {(solid == ordered).mean():.2f}")
```

</details>

In [ ]:
# 2. Energy in kelvin, without and with standardising
# FIXME

<details><summary {style="color:green;font-weight:bold"}> Click here to see solution to Task 4.2 </summary>

```python
# 2. Energy in kelvin, without and with standardising
E_K = 120 * E
for name, F_K in [("not standardised", np.c_[CN, E_K]),
                  ("standardised", StandardScaler().fit_transform(np.c_[CN, E_K]))]:
    s = solid_like(KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(F_K))
    print(f"{name:17s}: agreement {(s == ordered).mean():.2f}")
s_E = solid_like(KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(E[:, None]))
print(f"energy alone     : agreement {(s_E == ordered).mean():.2f}")
print("standard deviations: CN", CN.std().round(2), "; E_K", E_K.std().round(1), "K")
```

</details>

*Your answers to questions 1 and 2 here.*

<details><summary {style="color:green;font-weight:bold"}> Click here to see answers to questions 1 and 2 (Tasks 4) </summary>

About 63 % of atoms are solid-like, and the clustering agrees with $|\psi_6| > 0.6$ for about 73 % of atoms; most disagreements are at the crystal–liquid boundaries and in ordered patches inside the liquid. With the energy in kelvin and no standardising, the agreement drops to about 0.67, exactly the value for clustering on energy alone. The energy now has a standard deviation of about 44 K against 0.7 for CN, so the distances k-means computes are almost entirely energy differences and CN is ignored. After standardising we are back at 0.73. The physics has not changed, only the units: without scaling, the units decide what the algorithm pays attention to.

</details>

# 5. How many clusters? <a id="5-k"></a>

k-means needs $K$. Two common checks:
* the **within-cluster spread** (`KMeans(...).inertia_`: the sum of squared distances of the points to their centroid) always falls as $K$ grows; look for the **elbow** where adding a cluster stops helping much;
* the **silhouette score** (`silhouette_score(F, labels)`, between −1 and 1) is high when each point is much closer to its own cluster than to the next one.

## Tasks 5 <a id="tasks-5"></a>
1. For $K = 1$ to 6, run k-means on the standardised features `F` and store the inertia; for $K \geq 2$ also store the silhouette score. Plot both against $K$.
2. Which $K$ do the plots suggest? Run $K = 3$ and print, for each cluster, its size, mean CN, mean energy and the fraction of ordered atoms. What has the third cluster picked out?

In [ ]:
# 1. Elbow and silhouette
# FIXME

<details><summary {style="color:green;font-weight:bold"}> Click here to see solution to Task 5.1 </summary>

```python
# 1. Elbow and silhouette
Ks = range(1, 7)
inertia, silhouette = [], []
for k in Ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(F)
    inertia.append(km.inertia_)
    silhouette.append(silhouette_score(F, km.labels_) if k > 1 else np.nan)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(Ks, inertia, "o-", color=BLUE); axes[0].set_ylabel("within-cluster spread (inertia)")
axes[1].plot(Ks, silhouette, "o-", color=BLUE); axes[1].set_ylabel("silhouette score")
for ax in axes:
    ax.set_xlabel("K")
plt.show()
print("silhouette:", np.round(silhouette, 2))
```

</details>

In [ ]:
# 2. What does K = 3 find?
# FIXME

<details><summary {style="color:green;font-weight:bold"}> Click here to see solution to Task 5.2 </summary>

```python
# 2. What does K = 3 find?
labels3 = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(F)
for c in np.argsort([E[labels3 == c].mean() for c in range(3)]):
    m = labels3 == c
    print(f"cluster of {m.sum():3d} atoms: mean CN {CN[m].mean():.2f}, mean E {E[m].mean():.2f}, "
          f"ordered {ordered[m].mean():.0%}")
```

</details>

*Your answers to question 2 here.*

<details><summary {style="color:green;font-weight:bold"}> Click here to see answers to question 2 (Tasks 5) </summary>

The biggest drop in inertia is from $K = 1$ to 2, and the silhouette score is highest at $K = 2$ (about 0.58), so the data support two clusters. But $K = 3$ is almost as good (about 0.57), and the score only falls to about 0.55 for larger $K$, so the evidence for exactly two clusters is not strong. $K = 3$ splits off a small group of atoms with fewer neighbours (CN about 4) and higher energy, the most open part of the liquid, and leaves an intermediate group around CN = 5. The plots guide the choice of $K$, but whether a cluster means something (a phase, an interface, or nothing at all) is a chemistry question.

</details>

# 6. DBSCAN <a id="6-dbscan"></a>

DBSCAN does not need $K$. A point is a **core point** if it has at least `min_samples` points (itself included) within a distance `eps`; core points close to each other form a cluster, and points that are neither core points nor close to one are labelled **noise** (label −1). It finds regions of high density separated by regions of low density.

## Tasks 6 <a id="tasks-6"></a>
1. Run `DBSCAN(eps=0.3, min_samples=5)` on the standardised features `F`. How many clusters and noise points do you get? Print the CN values that occur in each cluster. What has DBSCAN found, and why?
2. The function below replaces the hard cutoff by a **smooth** one: a neighbour counts fully below $1.3\sigma$, not at all above $1.7\sigma$, and partly in between. Compute the smooth CN, standardise `np.c_[CN_smooth, E]` and run DBSCAN again with the same settings. How many clusters now? Compare with k-means.

In [ ]:
def smooth_cn(r, r_on=1.3, r_off=1.7):
    """Coordination number with a smooth cutoff: weight 1 below r_on, 0 above r_off,
    and a cosine switch in between."""
    x = np.clip((r - r_on) / (r_off - r_on), 0, 1)
    return (0.5 * (1 + np.cos(np.pi * x))).sum(axis=1)

In [ ]:
# 1. DBSCAN on (CN, E)
# FIXME

<details><summary {style="color:green;font-weight:bold"}> Click here to see solution to Task 6.1 </summary>

```python
# 1. DBSCAN on (CN, E)
db = DBSCAN(eps=0.3, min_samples=5).fit_predict(F)
print("clusters:", len(set(db) - {-1}), "; noise points:", (db == -1).sum())
for c in sorted(set(db) - {-1}):
    print(f"cluster {c}: {np.sum(db == c):3d} atoms, CN values {np.unique(CN[db == c])}")
```

</details>

In [ ]:
# 2. DBSCAN with a smooth coordination number
# FIXME

<details><summary {style="color:green;font-weight:bold"}> Click here to see solution to Task 6.2 </summary>

```python
# 2. DBSCAN with a smooth coordination number
CN_smooth = smooth_cn(r)
F_smooth = StandardScaler().fit_transform(np.c_[CN_smooth, E])
db_s = DBSCAN(eps=0.3, min_samples=5).fit_predict(F_smooth)
print("clusters:", len(set(db_s) - {-1}), "; noise points:", (db_s == -1).sum())

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(CN, E, s=10, alpha=0.4, color=BLUE); axes[0].set_xlabel("CN (hard cutoff)")
axes[1].scatter(CN_smooth, E, s=10, alpha=0.4, color=BLUE); axes[1].set_xlabel("CN (smooth cutoff)")
for ax in axes:
    ax.set_ylabel("potential energy / epsilon")
plt.show()
```

</details>

*Your answers to questions 1 and 2 here.*

<details><summary {style="color:green;font-weight:bold"}> Click here to see answers to questions 1 and 2 (Tasks 6) </summary>

With the hard cutoff DBSCAN finds about four clusters, and each one contains a single CN value (4, 5 or 6). Because CN is a whole number, the points lie on vertical stripes in feature space with empty gaps between them, and DBSCAN, which looks for gaps, finds the stripes. These "clusters" come from the descriptor, not from the physics. With the smooth CN the stripes merge and DBSCAN finds a single cluster plus a few noise points: in this feature space there is no low-density gap between solid-like and liquid-like atoms (a dense liquid next to a crystal, with interfaces in between), so there is nothing for DBSCAN to separate. k-means splits the data anyway, because we told it $K = 2$. Neither answer is wrong; they make different assumptions about what a cluster is, and the representation decides which assumption fits.

</details>

# 7. Key points <a id="7-key-points"></a>

* Descriptors built from distances and angles ($g(r)$, coordination number, energy, $|\psi_6|$) describe each atom's surroundings and do not change when the system is moved, rotated or renumbered.
* The **cutoff** is a choice: the first minimum of $g(r)$ is a good place for it. A hard cutoff gives a descriptor that jumps; smooth cutoffs avoid this.
* Different descriptors carry different information: CN and energy largely overlap, $|\psi_6|$ adds the angles.
* **k-means** needs standardised features (otherwise the units decide) and a choice of $K$; elbow and silhouette plots help, chemistry decides.
* **DBSCAN** finds dense regions separated by gaps. It does not need $K$, but it finds gaps that the descriptor creates (integer CN) and finds nothing when there is no gap.

**Questions to think about:**
1. Would you expect the same descriptors to work for a 3D LJ crystal? What would replace $|\psi_6|$?
2. You run the same analysis on a snapshot of a pure liquid. What would k-means with $K = 2$ return, and how would the silhouette score tell you that something is wrong?